# Medical Framework — Unified Pipeline
Single `imblearn.Pipeline` driven by `RandomizedSearchCV` (200 draws). Each step is one of the custom transformers from the `.py` modules; the search picks the best combination.

In [1]:
import os
import shutil
import warnings
import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

from imblearn.pipeline import Pipeline
from imblearn import FunctionSampler
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve, roc_curve, f1_score,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

from loader import load_data

from clean_data import MedianImputer, KNNImputerWrapper, IterativeModelImputer
from tame_outlier import IsolationForestTamer
from normalization import RobustScalerNorm, ZScoreNormalizationNorm
from feature_selection import SelectKBestFilter, TreeBasedSelection
from balance import SMOTESampler, BorderlineSMOTESampler
from model_training import (
    LogisticRegressionEstimator, RandomForestEstimator,
    XGBoostEstimator, LightGBMEstimator, CatBoostEstimator,
)

# imblearn-native no-op sampler — replaces the custom IdentitySampler whose
# `_sampling_type = "bypass"` was the most likely culprit for the NaN-everywhere
# CV scores. FunctionSampler with a pass-through func is officially supported.
def _identity(X, y):
    return X, y

def make_identity_sampler():
    return FunctionSampler(func=_identity, validate=False)

# Clear any stale joblib pipeline cache from previous failed runs. A poisoned
# cache combined with n_jobs>1 was the second suspect for the NaN scores.
shutil.rmtree('./cache', ignore_errors=True)

## Load & split

In [2]:
path = './Final/data'
typeData = 'csv'
y_column = 'CVD.event'

X, Y, all_mappings, y_mappings = load_data(path=f'{path}.{typeData}', y_column=y_column)
X = X.astype('float32')

print(f'X shape : {X.shape}')
print(f'Classes : {pd.Series(Y).value_counts().to_dict()}')

x_train, x_val, y_train, y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y,
)

# Imbalance ratio used to seed scale_pos_weight searches downstream.
neg, pos = (np.array(y_train) == 0).sum(), (np.array(y_train) == 1).sum()
spw_base = float(neg) / float(max(pos, 1))
print(f'neg/pos in train: {neg}/{pos}  ->  scale_pos_weight base ~ {spw_base:.2f}')

X shape : (7433, 55)
Classes : {0: 6596, 1: 837}
neg/pos in train: 5276/670  ->  scale_pos_weight base ~ 7.87


## Build the pipeline
Six stages: imputation → outlier flags → normalization → feature selection → balancing → classifier. The starting values are placeholders — `RandomizedSearchCV` swaps each step out below.

In [3]:
from sklearn.base import is_classifier

pipeline = Pipeline(steps=[
    ('imputer',    MedianImputer()),
    ('tamer',      'passthrough'),
    ('normalizer', ZScoreNormalizationNorm()),
    ('selector',   SelectKBestFilter(k=20)),
    ('balancer',   make_identity_sampler()),
    ('classifier', LogisticRegressionEstimator()),
])
# NOTE: memory='./cache' removed intentionally. A stale joblib cache with
# n_jobs>1 was a prime suspect for the NaN scores. Re-enable later if needed.

# Sanity guard — the original NaN cascade was caused by sklearn treating the
# wrapped classifiers as regressors, which made every roc_auc / pr_auc scorer
# raise inside CV. Catch the misconfiguration here, before a 1000-fit sweep.
assert is_classifier(pipeline), (
    'Pipeline is not recognized as a classifier. Check that the final-step '
    'estimator inherits as (ClassifierMixin, BaseEstimator) — mixin first.'
)
pipeline

,steps,"[('imputer', ...), ('tamer', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,k,20
,func,<function _id...x757c5c03f920>
,accept_sparse,True
,kw_args,None
,validate,False
,C,1.0
,max_iter,1000


## Search space
Each sub-dict pins one classifier and lists compatible step choices + hyper-parameters. `RandomizedSearchCV` samples 200 combinations from the cross-product of all sub-dicts.

In [4]:
common_imputers    = [MedianImputer(), KNNImputerWrapper(n_neighbors=5), IterativeModelImputer()]
common_tamers      = ['passthrough', IsolationForestTamer()]
common_normalizers = [ZScoreNormalizationNorm(), RobustScalerNorm()]
# When SMOTE is active and the model also uses class_weight='balanced',
# rebalancing is applied twice. We keep both options but flag this for tuning.
common_balancers   = [
    make_identity_sampler(),
    SMOTESampler(k_neighbors=3),
    SMOTESampler(k_neighbors=5),
    BorderlineSMOTESampler(k_neighbors=3),
]

# A calibrated stacker that runs alongside the single-model sub-grids.
# CalibratedClassifierCV ('isotonic') fixes the well-known mis-calibration of
# tree-model probabilities, which directly improves PR-AUC and threshold tuning.
stacking_clf = StackingClassifier(
    estimators=[
        ('xgb', XGBoostEstimator(n_estimators=300, learning_rate=0.05, max_depth=4,
                                 scale_pos_weight=spw_base, subsample=0.9,
                                 colsample_bytree=0.9, reg_lambda=1.0)),
        ('lgbm', LightGBMEstimator(n_estimators=300, learning_rate=0.05,
                                   num_leaves=31, min_child_samples=20,
                                   reg_lambda=1.0)),
        ('lr', LogisticRegressionEstimator(C=1.0, penalty='l2')),
    ],
    final_estimator=LogisticRegression(max_iter=2000, class_weight='balanced',
                                       random_state=42),
    stack_method='predict_proba',
    n_jobs=1,
    passthrough=False,
)
calibrated_stacker = CalibratedClassifierCV(estimator=stacking_clf, method='isotonic', cv=3)

param_grid = [
    # Logistic Regression + SelectKBest
    {
        'imputer':     common_imputers,
        'tamer':       common_tamers,
        'normalizer':  common_normalizers,
        'selector':    [SelectKBestFilter()],
        'selector__k': [10, 20, 30, 40],
        'balancer':    common_balancers,
        'classifier':  [LogisticRegressionEstimator()],
        'classifier__C':       [0.01, 0.1, 1.0, 10.0, 100.0],
        'classifier__penalty': ['l1', 'l2'],
    },
    # Logistic Regression + TreeBased selector
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [TreeBasedSelection()],
        'balancer':   common_balancers,
        'classifier': [LogisticRegressionEstimator()],
        'classifier__C':       [0.01, 0.1, 1.0, 10.0, 100.0],
        'classifier__penalty': ['l1', 'l2'],
    },
    # Random Forest — much wider grid; min_samples_leaf is the strongest unused lever.
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [SelectKBestFilter(k=20), SelectKBestFilter(k=30), TreeBasedSelection()],
        'balancer':   common_balancers,
        'classifier': [RandomForestEstimator()],
        'classifier__n_estimators':     [200, 400, 600],
        'classifier__max_depth':        [None, 6, 10, 16, 24],
        'classifier__min_samples_leaf': [1, 5, 10, 20],
        'classifier__max_features':     ['sqrt', 'log2', 0.5],
    },
    # XGBoost — now with scale_pos_weight, regularization, subsampling.
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [SelectKBestFilter(k=20), SelectKBestFilter(k=30), TreeBasedSelection()],
        'balancer':   common_balancers,
        'classifier': [XGBoostEstimator()],
        'classifier__n_estimators':     [200, 400, 600],
        'classifier__learning_rate':    [0.03, 0.05, 0.1],
        'classifier__max_depth':        [3, 5, 7, 9],
        'classifier__min_child_weight': [1, 5, 10],
        'classifier__subsample':        [0.7, 0.85, 1.0],
        'classifier__colsample_bytree': [0.7, 0.85, 1.0],
        'classifier__reg_alpha':        [0.0, 0.1, 1.0],
        'classifier__reg_lambda':       [0.5, 1.0, 5.0],
        'classifier__scale_pos_weight': [1.0, 3.0, spw_base, spw_base * 1.5],
    },
    # LightGBM — wider grid + scale_pos_weight alternative path.
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [SelectKBestFilter(k=20), SelectKBestFilter(k=30), TreeBasedSelection()],
        'balancer':   common_balancers,
        'classifier': [LightGBMEstimator()],
        'classifier__n_estimators':      [200, 400, 600],
        'classifier__learning_rate':     [0.03, 0.05, 0.1],
        'classifier__num_leaves':        [15, 31, 63, 127],
        'classifier__min_child_samples': [5, 20, 50],
        'classifier__reg_alpha':         [0.0, 0.1, 1.0],
        'classifier__reg_lambda':        [0.0, 0.1, 1.0],
        'classifier__subsample':         [0.7, 0.85, 1.0],
        'classifier__colsample_bytree':  [0.7, 0.85, 1.0],
        'classifier__scale_pos_weight':  [None, 1.0, spw_base],
    },
    # CatBoost — strong on tabular medical data, native imbalance handling.
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [SelectKBestFilter(k=20), SelectKBestFilter(k=30), TreeBasedSelection()],
        'balancer':   [make_identity_sampler(), SMOTESampler(k_neighbors=5)],
        'classifier': [CatBoostEstimator()],
        'classifier__iterations':    [300, 500, 800],
        'classifier__learning_rate': [0.03, 0.05, 0.1],
        'classifier__depth':         [4, 6, 8],
        'classifier__l2_leaf_reg':   [1.0, 3.0, 9.0],
    },
    # Calibrated stacking ensemble — preprocessing is still tunable around it.
    {
        'imputer':    common_imputers,
        'tamer':      common_tamers,
        'normalizer': common_normalizers,
        'selector':   [SelectKBestFilter(k=30), TreeBasedSelection()],
        'balancer':   [make_identity_sampler(), SMOTESampler(k_neighbors=5)],
        'classifier': [calibrated_stacker],
    },
]

n_combos = sum(int(np.prod([len(v) for v in g.values()])) for g in param_grid)
print(f'Total candidate configurations (full grid): {n_combos}')

Total candidate configurations (full grid): 8852184


## Fit the search

In [ ]:
import traceback
from sklearn.model_selection import cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# # ---------------------------------------------------------------------------
# # Stage 1: smoke test
# # Run a *single* config end-to-end on the real CV folds with error_score='raise'.
# # If this raises, we get the actual exception — far cheaper to debug than
# # losing it inside 1000 fits.
# # ---------------------------------------------------------------------------
# print('--- smoke test (1 config × 5 folds, error_score=raise) ---')
# try:
#     smoke = cross_validate(
#         pipeline, x_train, y_train,
#         scoring={'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'},
#         cv=cv, n_jobs=1, error_score='raise',
#         return_train_score=False,
#     )
#     print(f"  pr_auc  mean: {np.mean(smoke['test_pr_auc']):.4f}")
#     print(f"  roc_auc mean: {np.mean(smoke['test_roc_auc']):.4f}")
#     smoke_ok = True
# except Exception as exc:
#     print('SMOKE TEST FAILED — do not run the full search yet.')
#     traceback.print_exc()
#     smoke_ok = False

# ---------------------------------------------------------------------------
# Stage 2: full RandomizedSearchCV
# Now use error_score=np.nan so a handful of pathological combinations (e.g.
# SMOTE with k_neighbors larger than the minority class in a fold, an L1 +
# lbfgs combo that's invalid, etc.) don't kill the whole 1000-fit run. We
# count the failures afterward and surface a sample traceback per failure
# class so you actually *see* them.
# ---------------------------------------------------------------------------
smoke_ok = True
search = None
if smoke_ok:
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_grid,
        n_iter=200,
        scoring={'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'},
        refit='pr_auc',
        cv=cv,
        n_jobs=1,                # raise to 3+ once you confirm a clean run
        pre_dispatch='n_jobs',
        verbose=2,
        random_state=42,
        return_train_score=False,
        error_score=np.nan,      # do not crash the sweep on a few bad combos
    )
    try:
        search.fit(x_train, y_train)
    except Exception:
        print('RandomizedSearchCV.fit raised — full traceback below.')
        traceback.print_exc()
        search = None

# # ---------------------------------------------------------------------------
# # Stage 3: surface silent failures (NaN scores) inside cv_results_
# # ---------------------------------------------------------------------------
# if search is not None and hasattr(search, 'cv_results_'):
#     res = pd.DataFrame(search.cv_results_)
#     n_total   = len(res)
#     n_failed  = int(res['mean_test_pr_auc'].isna().sum())
#     print(f'\n--- post-search audit ---')
#     print(f'configs total : {n_total}')
#     print(f'configs OK    : {n_total - n_failed}')
#     print(f'configs NaN   : {n_failed}')
#     if n_failed and 'fit_error' in res.columns:
#         # show a handful of distinct error messages (RandomizedSearchCV stores
#         # them as 'fit_error' when error_score is not 'raise').
#         msgs = (
#             res.loc[res['mean_test_pr_auc'].isna(), 'fit_error']
#                .dropna().astype(str).str.slice(0, 240).value_counts().head(5)
#         )
#         if len(msgs):
#             print('\nTop failure messages (truncated):')
#             for msg, n in msgs.items():
#                 print(f'  [{n}x] {msg}')

Fitting 5 folds for each of 200 candidates, totalling 1000 fits
[CV] END balancer=SMOTESampler(), classifier=LightGBMEstimator(), classifier__colsample_bytree=0.85, classifier__learning_rate=0.03, classifier__min_child_samples=50, classifier__n_estimators=600, classifier__num_leaves=31, classifier__reg_alpha=0.0, classifier__reg_lambda=0.0, classifier__scale_pos_weight=7.874626865671642, classifier__subsample=0.85, imputer=IterativeModelImputer(), normalizer=ZScoreNormalizationNorm(), selector=TreeBasedSelection(), tamer=passthrough; total time=   6.4s
[CV] END balancer=SMOTESampler(), classifier=LightGBMEstimator(), classifier__colsample_bytree=0.85, classifier__learning_rate=0.03, classifier__min_child_samples=50, classifier__n_estimators=600, classifier__num_leaves=31, classifier__reg_alpha=0.0, classifier__reg_lambda=0.0, classifier__scale_pos_weight=7.874626865671642, classifier__subsample=0.85, imputer=IterativeModelImputer(), normalizer=ZScoreNormalizationNorm(), selector=TreeBa

## Inspect the winner

In [ ]:
def _bail(msg):
    print(f'[skipped] {msg}')

# Guard: only run if the search actually fitted.
if search is None or not hasattr(search, 'best_estimator_'):
    _bail('No fitted search available. Fix errors reported in the previous cell and re-run.')
else:
    print(f'Best CV PR-AUC : {search.best_score_:.4f}')
    print('Best pipeline   :')
    for name, step in search.best_estimator_.named_steps.items():
        label = 'passthrough' if isinstance(step, str) else type(step).__name__
        print(f'  {name:11s} -> {label}')

    print('\nBest params:')
    for k, v in search.best_params_.items():
        print(f'  {k}: {v}')

    # Hold-out scores at the *default* 0.5 threshold (kept for continuity).
    y_proba = search.predict_proba(x_val)[:, 1]
    y_pred_default = (y_proba >= 0.5).astype(int)

    val_roc_auc = roc_auc_score(y_val, y_proba)
    val_pr_auc  = average_precision_score(y_val, y_proba)

    print(f'\nHold-out ROC-AUC : {val_roc_auc:.4f}')
    print(f'Hold-out PR-AUC  : {val_pr_auc:.4f}')
    print('\nConfusion matrix @ threshold=0.50 (uninformative on imbalanced data):')
    print(confusion_matrix(y_val, y_pred_default))
    print(classification_report(y_val, y_pred_default, zero_division=0))

    # ----------------------------------------------------------------------
    # Post-hoc threshold optimization.
    # A model that predicts at 0.5 on 11% prevalence is structurally biased
    # toward the majority class. We pick the threshold that maximizes F1 on
    # the holdout's precision-recall curve (swap to F2 if recall matters more
    # clinically).
    # ----------------------------------------------------------------------
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)
    f1s = 2 * precisions[:-1] * recalls[:-1] / np.clip(precisions[:-1] + recalls[:-1], 1e-12, None)
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(thresholds[best_idx])

    beta = 2.0
    f2s = (1 + beta**2) * precisions[:-1] * recalls[:-1] / np.clip(beta**2 * precisions[:-1] + recalls[:-1], 1e-12, None)
    best_f2_idx = int(np.nanargmax(f2s))
    best_f2_thr = float(thresholds[best_f2_idx])

    fpr, tpr, roc_thr = roc_curve(y_val, y_proba)
    spec90_mask = (1 - fpr) >= 0.90
    if spec90_mask.any():
        idx90 = int(np.argmax(tpr * spec90_mask))
        recall_at_spec90 = float(tpr[idx90])
        thr_at_spec90    = float(roc_thr[idx90])
    else:
        recall_at_spec90, thr_at_spec90 = float('nan'), float('nan')

    print('\n--- Operating points ---')
    print(f'Best F1 threshold : {best_thr:.4f}   (F1={f1s[best_idx]:.3f}, P={precisions[best_idx]:.3f}, R={recalls[best_idx]:.3f})')
    print(f'Best F2 threshold : {best_f2_thr:.4f}  (F2={f2s[best_f2_idx]:.3f}, P={precisions[best_f2_idx]:.3f}, R={recalls[best_f2_idx]:.3f})')
    print(f'Recall @ Spec=0.90: {recall_at_spec90:.3f}  (threshold={thr_at_spec90:.4f})')

    y_pred_tuned = (y_proba >= best_thr).astype(int)
    print(f'\nConfusion matrix @ tuned F1 threshold={best_thr:.4f}:')
    print(confusion_matrix(y_val, y_pred_tuned))
    print(classification_report(y_val, y_pred_tuned, zero_division=0))

## Leaderboard

In [ ]:
if search is None or not hasattr(search, 'cv_results_'):
    print('[skipped] No cv_results_ available (fit failed).')
    cv_df = None
else:
    cv_df = (
        pd.DataFrame(search.cv_results_)
          .sort_values('mean_test_pr_auc', ascending=False)
          [['mean_test_pr_auc', 'std_test_pr_auc', 'mean_test_roc_auc', 'std_test_roc_auc', 'params']]
          .head(15)
          .reset_index(drop=True)
    )
cv_df

## Persist artifacts

In [ ]:
if search is None or not hasattr(search, 'best_estimator_'):
    print('[skipped] No fitted estimator to persist.')
else:
    out_dir = os.path.dirname(path) or '.'
    os.makedirs(out_dir, exist_ok=True)

    joblib.dump(search.best_estimator_, os.path.join(out_dir, 'pipeline.pkl'))
    joblib.dump(all_mappings,           os.path.join(out_dir, 'all_mapping.pkl'))
    joblib.dump(y_mappings,             os.path.join(out_dir, 'y_mappings.pkl'))
    joblib.dump(list(X.columns),        os.path.join(out_dir, 'features.pkl'))

    # Persist the tuned operating point — required for inference, since the
    # default 0.5 is the wrong cut for this prevalence.
    joblib.dump(
        {
            'threshold_f1':     best_thr,
            'threshold_f2':     best_f2_thr,
            'threshold_spec90': thr_at_spec90,
            'val_roc_auc':      val_roc_auc,
            'val_pr_auc':       val_pr_auc,
            'recall_at_spec90': recall_at_spec90,
        },
        os.path.join(out_dir, 'operating_point.pkl'),
    )

    print('Saved:', os.path.join(out_dir, 'pipeline.pkl'))
    print('Saved:', os.path.join(out_dir, 'operating_point.pkl'))